导入数据库查询和战斗模型

In [2]:
import pandas as pd
from IPython.display import display

from tft_simulator.database.queries import (
    read_connection,
    get_champion,
)
from tft_simulator.models import ChampionStats, CombatState

# 这里是数据库版本记录ID，不是星级。
VERSION_ID = 1

print("P4模块导入成功")

P4模块导入成功


读取安妮和奥拉夫

In [3]:
with read_connection() as connection:
    record_a = get_champion(connection, VERSION_ID, "TFT10_Annie")
    record_b = get_champion(connection, VERSION_ID, "TFT10_Olaf")

if record_a is None or record_b is None:
    raise ValueError("没有找到指定英雄，请检查数据库中的版本和英雄ID。")

stats_a = ChampionStats.from_record(record_a)
stats_b = ChampionStats.from_record(record_b)

display(pd.DataFrame([
    {
        "name": stats.name,
        "max_hp": stats.max_hp,
        "attack_damage": stats.attack_damage,
        "attack_speed": stats.attack_speed,
        "armor": stats.armor,
        "magic_resist": stats.magic_resist,
    }
    for stats in (stats_a, stats_b)
]))

,name,max_hp,attack_damage,attack_speed,armor,magic_resist
0,Annie,450.0,40.0,0.65,20.0,20.0
1,Olaf,650.0,55.0,0.65,35.0,35.0


创建战斗状态，演示扣血

In [4]:
# 每场战斗都创建新状态，开始时使用完整生命值和初始法力。
state_a = CombatState(stats_a)
state_b = CombatState(stats_b)

print("目标英雄：", state_b.stats.name)
print("受伤前生命值：", state_b.current_hp)

# 手动输入100点“已经结算好的伤害”，这里只演示状态变化。
# 这不是Annie一次普通攻击或技能的真实伤害。
actual_damage = state_b.take_damage(100.0)

print("实际扣血：", actual_damage)
print("受伤后生命值：", state_b.current_hp)
print("是否存活：", state_b.is_alive)

# 基础属性和原始查询结果都不应受到影响。
print("基础生命值：", stats_b.max_hp)
print("原始记录生命值：", record_b["hp"])

# 同一个英雄进入另一场战斗，应从完整状态重新开始。
next_battle_state = CombatState(stats_b)
print("下一场初始生命值：", next_battle_state.current_hp)

目标英雄： Olaf
受伤前生命值： 650.0
实际扣血： 100.0
受伤后生命值： 550.0
是否存活： True
基础生命值： 650.0
原始记录生命值： 650.0
下一场初始生命值： 650.0


导入函数观察例子

In [5]:
from tft_simulator.combat import (
    physical_damage,
    magic_damage,
    true_damage,
)

# 以下是人工数值例子，用于理解公式，不代表某个英雄的技能。
physical_result = physical_damage(100.0, armor=100.0)
magic_result = magic_damage(100.0, magic_resist=50.0)
true_result = true_damage(100.0)

print(f"100物理伤害，对100护甲：{physical_result:.4f}")
print(f"100魔法伤害，对50魔抗：{magic_result:.4f}")
print(f"100真实伤害：{true_result:.4f}")

100物理伤害，对100护甲：50.0000
100魔法伤害，对50魔抗：66.6667
100真实伤害：100.0000


模拟用安妮和奥拉夫的属性完成一次扣血

In [6]:
target_state = CombatState(stats_b)

#不触发暴击、技能、被动或装备效果
raw_damage = stats_a.attack_damage

damage_after_armor = physical_damage(
    raw_damage,
    armor=target_state.stats.armor,
)

hp_before = target_state.current_hp

actual_damage = target_state.take_damage(damage_after_armor)

print("攻击者：", stats_a.name)
print("目标：", target_state.stats.name)
print(f"原始伤害：{raw_damage:.4f}")
print(f"护甲计算后伤害：{damage_after_armor:.4f}")
print(f"实际扣血：{actual_damage:.4f}")
print(f"目标生命值：{hp_before:.4f} → {target_state.current_hp:.4f}")

攻击者： Annie
目标： Olaf
原始伤害：40.0000
护甲计算后伤害：29.6296
实际扣血：29.6296
目标生命值：650.0000 → 620.3704


安妮、奥拉夫攻速转化成攻击间隔时间

In [8]:
from tft_simulator.combat.timing import attack_interval, attack_times

for stats in (stats_a, stats_b):
    interval = attack_interval(stats.attack_speed)

    print(
        f"{stats.name}："
        f"攻速约 {stats.attack_speed:.4f} 次/秒，"
        f"攻击间隔约 {interval:.4f} 秒"
    )

Annie：攻速约 0.6500 次/秒，攻击间隔约 1.5385 秒
Olaf：攻速约 0.6500 次/秒，攻击间隔约 1.5385 秒


安妮十秒内攻击时间表

In [9]:
duration = 10.0
times = attack_times(
    attack_speed=stats_a.attack_speed,
    duration=duration,
)

timeline_df = pd.DataFrame([
    {
        "attack_number": number,
        "time_seconds": time,
    }
    for number, time in enumerate(times, start=1)
])

print("英雄：", stats_a.name)
print("观察时长：", duration, "秒")
print("计划攻击次数：", len(times))

with pd.option_context("display.precision", 4):
    display(timeline_df)

英雄： Annie
观察时长： 10.0 秒
计划攻击次数： 6


,attack_number,time_seconds
0,1,1.5385
1,2,3.0769
2,3,4.6154
3,4,6.1538
4,5,7.6923
5,6,9.2308


运行完整战斗

In [13]:
from tft_simulator.combat.engine import run_duel

# 每次调用都会在内部创建全新的战斗状态。
result = run_duel(
    stats_a,
    stats_b,
    max_duration=60.0,
)

summary_df = pd.DataFrame([result.summary()])

with pd.option_context("display.precision", 4):
    display(summary_df)

,winner,winner_name,reason,duration,hp_a,hp_b,attacks_a,attacks_b,damage_a,damage_b
0,B,Olaf,death,15.3846,0.0,353.7037,10,10,296.2963,450.0


日志

In [14]:
battle_log = pd.DataFrame(result.log_records())
display_columns = [
    "time",
    "event_type",
    "actor",
    "target",
    "damage",
    "hp_a",
    "hp_b",
]
print("日志记录数：", len(battle_log))
with pd.option_context("display.precision", 4):
    display(battle_log[display_columns])

日志记录数： 23


,time,event_type,actor,target,damage,hp_a,hp_b
0,0.0000,battle_start,NaN,NaN,0.0000,450.0000,650.0000
1,1.5385,attack,Annie,Olaf,29.6296,404.1667,620.3704
2,1.5385,attack,Olaf,Annie,45.8333,404.1667,620.3704
3,3.0769,attack,Annie,Olaf,29.6296,358.3333,590.7407
4,3.0769,attack,Olaf,Annie,45.8333,358.3333,590.7407
5,4.6154,attack,Annie,Olaf,29.6296,312.5000,561.1111
6,4.6154,attack,Olaf,Annie,45.8333,312.5000,561.1111
7,6.1538,attack,Annie,Olaf,29.6296,266.6667,531.4815
8,6.1538,attack,Olaf,Annie,45.8333,266.6667,531.4815
9,7.6923,attack,Annie,Olaf,29.6296,220.8333,501.8519


观察超时平局

In [15]:
#时间缩短至五秒
timeout_result = run_duel(
    stats_a,
    stats_b,
    max_duration=5.0,
)

with pd.option_context("display.precision", 4):
    display(pd.DataFrame([timeout_result.summary()]))

,winner,winner_name,reason,duration,hp_a,hp_b,attacks_a,attacks_b,damage_a,damage_b
0,draw,None,timeout,5.0,312.5,561.1111,3,3,88.8889,137.5
